# Google Colab 3 - Logistic Regression + Evaluasi Perbandingan

**Orang 3** bertanggung jawab untuk normalisasi/standardisasi fitur, implementasi Logistic Regression, evaluasi model, tabel perbandingan, grafik perbandingan, analisis komparasi, dan kesimpulan.

## 1. Import Library

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load Dataset

Jika dijalankan di Colab, upload `heart.csv` atau pastikan file tersedia di folder `data/heart.csv`.

In [ ]:
def load_dataset():
    candidates = [Path('data/heart.csv'), Path('heart.csv')]
    for path in candidates:
        if path.exists():
            print(f'Dataset digunakan: {path}')
            return pd.read_csv(path)
    raise FileNotFoundError('Dataset tidak ditemukan. Upload heart.csv atau simpan di folder data/.')

df = load_dataset()
df.head()

## 3. Data Checking dan Cleaning

In [ ]:
print('Jumlah baris dan kolom awal:', df.shape)
display(df.describe().T)

missing_df = pd.DataFrame({
    'Kolom': df.isnull().sum().index,
    'Jumlah Missing Value': df.isnull().sum().values
})
display(missing_df)

duplicate_count = df.duplicated().sum()
print('Jumlah data duplikat:', duplicate_count)
if duplicate_count > 0:
    df = df.drop_duplicates()
    print('Data duplikat berhasil dihapus.')

print('Ukuran dataset setelah cleaning:', df.shape)

## 4. Split Fitur dan Target

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print('Ukuran fitur X:', X.shape)
print('Ukuran target y:', y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Jumlah data training:', X_train.shape[0])
print('Jumlah data testing:', X_test.shape[0])

## 5. Standardisasi Fitur

Logistic Regression sensitif terhadap skala fitur, sehingga fitur distandardisasi menggunakan `StandardScaler`. Scaler hanya di-fit pada data training agar tidak terjadi data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:5]

## 6. Implementasi Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
print('Hasil prediksi Logistic Regression:')
print(y_pred_lr)

## 7. Evaluasi Logistic Regression

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

eval_lr = pd.DataFrame({
    'Model': ['Logistic Regression'],
    'Accuracy': [accuracy_lr],
    'Precision': [precision_lr],
    'Recall': [recall_lr],
    'F1-Score': [f1_lr]
})

display(eval_lr.round(4))
print('Classification Report Logistic Regression:')
print(classification_report(y_test, y_pred_lr))

## 8. Confusion Matrix Logistic Regression

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
print('Confusion Matrix Logistic Regression:')
print(cm_lr)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_logistic_regression.png', dpi=300)
plt.show()

## 9. Tabel Perbandingan Model

Ganti nilai Decision Tree dan Random Forest dengan hasil final dari anggota 1 dan 2 jika terdapat perbedaan. Nilai awal di bawah dihitung menggunakan cleaning duplikat, split, dan `random_state` yang sama.

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': 'Decision Tree',
        'Accuracy': 0.721311,
        'Precision': 0.750000,
        'Recall': 0.727273,
        'F1-Score': 0.738462,
    },
    {
        'Model': 'Random Forest',
        'Accuracy': 0.754098,
        'Precision': 0.764706,
        'Recall': 0.787879,
        'F1-Score': 0.776119,
    },
    {
        'Model': 'Logistic Regression',
        'Accuracy': accuracy_lr,
        'Precision': precision_lr,
        'Recall': recall_lr,
        'F1-Score': f1_lr,
    }
])

comparison_df.to_csv(OUTPUT_DIR / 'tabel_perbandingan_model.csv', index=False)
display(comparison_df.round(4))

## 10. Grafik Perbandingan Accuracy, Precision, Recall, dan F1-Score

In [ ]:
comparison_melted = comparison_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    var_name='Metrik',
    value_name='Nilai'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_melted, x='Metrik', y='Nilai', hue='Model')
plt.title('Perbandingan Evaluasi Model')
plt.ylim(0, 1)
plt.ylabel('Nilai')
plt.xlabel('Metrik Evaluasi')
plt.legend(title='Model')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grafik_perbandingan_model.png', dpi=300)
plt.show()

## 11. Menentukan Model Terbaik

In [ ]:
best_model = comparison_df.sort_values(by=['F1-Score', 'Accuracy'], ascending=False).iloc[0]

print('Model terbaik berdasarkan F1-score dan Accuracy:')
print(f"Model    : {best_model['Model']}")
print(f"Accuracy : {best_model['Accuracy']:.4f}")
print(f"Precision: {best_model['Precision']:.4f}")
print(f"Recall   : {best_model['Recall']:.4f}")
print(f"F1-Score : {best_model['F1-Score']:.4f}")

## 12. Analisis Komparasi

Decision Tree memiliki performa yang cukup baik dan mudah diinterpretasikan, tetapi cenderung lebih sederhana sehingga hasilnya masih dapat kalah dibandingkan model ensemble. Random Forest meningkatkan performa Decision Tree karena menggunakan banyak pohon keputusan, sehingga prediksi lebih stabil dan mampu mengurangi risiko overfitting dari satu pohon tunggal.

Logistic Regression menggunakan fitur yang sudah distandardisasi. Model ini cocok sebagai baseline linear karena sederhana, cepat, dan mudah dipahami. Namun, jika hubungan antar fitur dan target tidak sepenuhnya linear, performanya dapat berada di bawah model berbasis tree.

Berdasarkan tabel perbandingan, model terbaik ditentukan menggunakan F1-score sebagai metrik utama, kemudian Accuracy sebagai pembanding tambahan. F1-score dipilih karena dapat melihat keseimbangan antara precision dan recall.

## 13. Kesimpulan

Pada dataset Heart Disease, Logistic Regression perlu menggunakan standardisasi fitur karena model ini sensitif terhadap skala data. Setelah Logistic Regression dievaluasi dan dibandingkan dengan Decision Tree serta Random Forest, model terbaik dipilih berdasarkan F1-score tertinggi.

Untuk kasus prediksi penyakit jantung, F1-score penting karena evaluasi tidak hanya melihat jumlah prediksi benar secara umum, tetapi juga keseimbangan antara kemampuan model menemukan pasien sakit dan ketepatan prediksi sakit. Model dengan F1-score tertinggi lebih direkomendasikan untuk digunakan sebagai model akhir.